In [ ]:
# ─────────────────────────────────────────────────────────────
# TOOL 1: Search Flights
# In a real agent → calls Skyscanner / MakeMyTrip API
# Here           → simulated data
# ─────────────────────────────────────────────────────────────
def search_flights(source, destination):
    print(f"  🔍 Searching flights: {source} → {destination}...")
    return [
        {"airline": "IndiGo",    "price": 4500, "time": "06:00 AM", "duration": "2h 05m"},
        {"airline": "Air India", "price": 5200, "time": "09:00 AM", "duration": "2h 15m"},
        {"airline": "SpiceJet",  "price": 3900, "time": "11:30 AM", "duration": "2h 20m"},
        {"airline": "Vistara",   "price": 6100, "time": "07:45 AM", "duration": "2h 00m"},
    ]


# ─────────────────────────────────────────────────────────────
# TOOL 2: Check Weather
# In a real agent → calls OpenWeatherMap API
# Here           → simulated data
# ─────────────────────────────────────────────────────────────
def check_weather(city):
    print(f"  🌤️  Checking weather in {city}...")
    weather_db = {
        "Mumbai": {"temperature": 31, "condition": "Partly Cloudy", "flight_risk": "Low"},
        "Delhi":  {"temperature": 28, "condition": "Clear",          "flight_risk": "Low"},
    }
    return weather_db.get(city, {"temperature": 25, "condition": "Unknown", "flight_risk": "Unknown"})


# ─────────────────────────────────────────────────────────────
# TOOL 3: Calculate Best Budget Option
# Agent logic   → finds cheapest flight from list
# ─────────────────────────────────────────────────────────────
def calculate_budget(flights):
    print("  💰 Calculating cheapest option...")
    cheapest = min(flights, key=lambda x: x["price"])
    return cheapest


print("✅ Tools registered:")
print("   • search_flights(source, destination)")
print("   • check_weather(city)")
print("   • calculate_budget(flights)")

---
## 🧠 STEP 2 — Parse Intent

> 👉 **"The agent reads the user query and extracts what it needs to act on"**
>
> In a real agent → this is an **LLM call** (e.g. GPT / Claude)
> Here → we simulate it with Python logic

In [ ]:
import re

# ── The user's request ───────────────────────────────────────
user_query = "Find me the best flight from Delhi to Mumbai."

print("📥 User Query:", user_query)
print()


# ── Simulated intent parser ──────────────────────────────────
def parse_intent(query):
    """
    Extracts: source city, destination city, priority (price / time / comfort)
    In a real agent → replace this with an LLM call.
    """
    intent = {}

    # Extract source and destination
    match = re.search(r'from (\w+) to (\w+)', query, re.IGNORECASE)
    if match:
        intent["source"]      = match.group(1).title()
        intent["destination"] = match.group(2).title()
    else:
        intent["source"]      = "Delhi"
        intent["destination"] = "Mumbai"

    # Detect priority keyword
    if "cheap" in query.lower() or "budget" in query.lower():
        intent["priority"] = "price"
    elif "fast" in query.lower() or "quick" in query.lower():
        intent["priority"] = "time"
    else:
        intent["priority"] = "best_overall"  # default

    return intent


intent = parse_intent(user_query)

print("🔄 Parsing intent...")
print()
print("✅ Extracted Intent:")
print(f"   source       = {intent['source']}")
print(f"   destination  = {intent['destination']}")
print(f"   priority     = {intent['priority']}")

---
## 🔧 STEP 3 — Agent Calls the Tools

> 👉 **"The agent decides which tools to call and in what order"**
>
> This is the **agentic loop** — the heart of any AI agent.

In [ ]:
print("⚙️  Agent is working...")
print("─" * 45)

# ── Tool Call 1: Search Flights ──────────────────────────────
flights = search_flights(intent["source"], intent["destination"])
print(f"   → Found {len(flights)} flights\n")

# ── Tool Call 2: Check Destination Weather ───────────────────
weather = check_weather(intent["destination"])
print(f"   → Weather: {weather['condition']}, {weather['temperature']}°C, "
      f"Flight Risk: {weather['flight_risk']}\n")

# ── Tool Call 3: Find Cheapest Option ───────────────────────
cheapest = calculate_budget(flights)
print(f"   → Cheapest: {cheapest['airline']} @ ₹{cheapest['price']}")

print("─" * 45)
print("\n✅ All tool calls complete. Moving to scoring...")

---
## ⚖️ STEP 4 — Score & Compare

> 👉 **"The agent doesn't just pick the cheapest — it balances price, time, and risk"**
>
> This is called a **scoring function** with weights.

In [ ]:
# ── Scoring weights ──────────────────────────────────────────
# Price  : lower is better  → weight 0.50
# Time   : earlier is better → weight 0.30
# Risk   : low risk = bonus  → weight 0.20

def score_flights(flights, weather):
    """
    Assigns a score to each flight.
    Higher score = better option.
    """
    # Normalise prices (cheapest gets highest score)
    min_price = min(f["price"] for f in flights)
    max_price = max(f["price"] for f in flights)

    # Convert departure time to hour for sorting
    def to_hour(t):
        h, m = t.replace(" AM","").replace(" PM","").split(":")
        hour = int(h) + (12 if "PM" in t and h != "12" else 0)
        return hour

    min_hour = min(to_hour(f["time"]) for f in flights)
    max_hour = max(to_hour(f["time"]) for f in flights)

    weather_bonus = 5 if weather["flight_risk"] == "Low" else 0

    scored = []
    for f in flights:
        price_score = (1 - (f["price"] - min_price) / (max_price - min_price + 1)) * 50
        hour        = to_hour(f["time"])
        time_score  = (1 - (hour - min_hour) / (max_hour - min_hour + 1)) * 30
        total_score = round(price_score + time_score + weather_bonus, 1)

        scored.append({**f, "score": total_score})

    return sorted(scored, key=lambda x: x["score"], reverse=True)


ranked = score_flights(flights, weather)

print("📊 Flight Rankings:")
print(f"{'Rank':<5} {'Airline':<12} {'Price':>7} {'Time':>10} {'Duration':>10} {'Score':>7}")
print("─" * 58)
for i, f in enumerate(ranked, 1):
    tag = "  ← 🏆 BEST" if i == 1 else ""
    print(f"{i:<5} {f['airline']:<12} ₹{f['price']:>6} {f['time']:>10} {f['duration']:>10} {f['score']:>7}{tag}")

---
## 🏆 STEP 5 — Final Recommendation

> 👉 **"The agent gives a structured, justified answer — not just a text reply"**

In [ ]:
best = ranked[0]

print("=" * 50)
print("  ✈️  SMART TRAVEL AGENT — RECOMMENDATION")
print("=" * 50)
print(f"  🛫 Route      : {intent['source']} → {intent['destination']}")
print(f"  🏅 Best Flight: {best['airline']}")
print(f"  💰 Price      : ₹{best['price']}")
print(f"  🕐 Departure  : {best['time']}")
print(f"  ⏱️  Duration   : {best['duration']}")
print(f"  🌤️  Weather    : {weather['condition']}, {weather['temperature']}°C")
print(f"  ⚠️  Flight Risk: {weather['flight_risk']}")
print(f"  📈 Agent Score: {best['score']} / 85")
print("=" * 50)
print()
print(f"💬 Agent says:")
print(f'   "I recommend the {best["airline"]} flight at {best["time"]}.')
print(f'    It offers the best balance of price (₹{best["price"]})')
print(f'    and departure time. Weather in {intent["destination"]} is')
print(f'    {weather["condition"].lower()} with low flight risk. Have a great trip!"')

---
## 🚀 BONUS — Full Agent in One Function

> 👉 **"This is how you wrap everything into a single agent call"**
>
> Try changing the query below and re-run!

In [ ]:
def run_travel_agent(query: str):
    """End-to-end Smart Travel Agent."""

    print(f"\n📥 Query: {query}")
    print("─" * 50)

    # Step 1: Parse intent
    intent  = parse_intent(query)

    # Step 2 & 3: Call tools
    flights = search_flights(intent["source"], intent["destination"])
    weather = check_weather(intent["destination"])

    # Step 4: Score
    ranked  = score_flights(flights, weather)
    best    = ranked[0]

    # Step 5: Output
    print("─" * 50)
    print(f"✅ Best Flight : {best['airline']}")
    print(f"   Price       : ₹{best['price']}")
    print(f"   Departure   : {best['time']}  ({best['duration']})")
    print(f"   Weather     : {weather['condition']}, {weather['temperature']}°C")
    print(f"   Agent Score : {best['score']} / 85")
    print("─" * 50)


# ── Try different queries! ───────────────────────────────────
run_travel_agent("Find me the best flight from Delhi to Mumbai.")

---
## 📝 Summary — What We Learned

| Concept | What it means | In this notebook |
|---|---|---|
| **Tool** | A function the agent can call | `search_flights`, `check_weather`, `calculate_budget` |
| **Intent Parsing** | Understanding the user query | `parse_intent()` |
| **Agentic Loop** | Agent calls tools step by step | STEP 3 |
| **Scoring Function** | Comparing options with weights | `score_flights()` |
| **Structured Output** | Final justified answer | STEP 5 |

---

### 🔗 Real-World Upgrades
- Replace `search_flights()` → **Skyscanner / MakeMyTrip API**
- Replace `check_weather()` → **OpenWeatherMap API**
- Replace `parse_intent()` → **Claude / GPT LLM call**
- Add memory → agent remembers your past trips
- Add more tools → hotel booking, cab booking, currency converter

> 🎯 **That's the power of AI Agents — they don't just talk, they act!**